# Trade Execution

Runs execution strategies against the 5-minute `BINNED_DATA` bins and analyses the results (fill rate, implementation shortfall, cost).

All reusable logic lives in [`execution.py`](execution.py); this notebook is the driver. (`test_snowflake.ipynb` stays the data explorer.)

**Layers** (see `execution.py` docstrings for full detail):

| Function | What it does |
|---|---|
| `simulate_bin_fill` | scalar reference model for a single bin |
| `twap_schedule` | baseline strategy: split quantity evenly across the day's bins |
| `run_strategy` | run one order `(security, date, side, quantity)` → per-bin fills |
| `run_trade_list` | run many orders (the trade list) → per-bin fills with `order_id` |
| `summarise_fills` | collapse to one row per order with fill rate / IS / cost |


In [ ]:
import polars as pl
from execution import (
    twap_schedule,
    run_strategy,
    run_trade_list,
    summarise_fills,
)

pl.Config.set_tbl_cols(-1)
RAW = "data/raw"

trade_list = pl.read_csv(f"{RAW}/trade_list.csv")
trade_list.head()

## Load bins

For a quick demo we load bins for just a couple of instruments (one liquid, one illiquid). For a full back-test, read the whole table instead — it is ~13M rows / ~2.5 GB in memory:

```python
bins = pl.read_csv(f"{RAW}/binned_data.csv")
```

In [ ]:
demo_secs = ["GC2025V Comdty", "PA2022Z Comdty"]  # liquid gold, illiquid palladium

bins = (
    pl.scan_csv(f"{RAW}/binned_data.csv")
    .filter(pl.col("security").is_in(demo_secs))
    .collect()
)
bins.shape

## Single order

Take one real row from the trade list (one bucket for a security-day) and run it. `run_strategy` takes the bins for that `(security, date)` plus the row's `side` and `quantity`, and returns one row per bin.

In [ ]:
# pick one real order from the trade list (one row = one bucket for a security-day)
order = trade_list.filter(pl.col("security") == "GC2025V Comdty").row(0, named=True)
print(order)

day = bins.filter(
    (pl.col("security") == order["security"])
    & (pl.col("publication_date") == order["date"])
)

fills = run_strategy(day, side=order["side"], quantity=order["quantity"], strategy=twap_schedule)
fills.head()

In [ ]:
summarise_fills(fills)

## Run the trade list

The `TRADE_LIST` table has six rows per (security, date) — `small_buys`, `small_sells`, `medium_buys`, `medium_sells`, `large_buys`, `large_sells`. Pass `trade_list=` to run **one bucket at a time** (the usual back-test). Each order gets a unique `order_id` and is auto-tagged with its bucket.

In [ ]:
orders = trade_list.filter(pl.col("security").is_in(demo_secs))

# run a single bucket, e.g. medium_buys (one of the six)
all_fills = run_trade_list(bins, orders, trade_list="medium_buys", strategy=twap_schedule)
all_fills.shape

In [ ]:
summary = summarise_fills(all_fills)
summary.sort("security", "side").select(
    "security", "side", "trade_list", "order_quantity", "fill_rate",
    "participation_overall", "avg_fill_price",
    "exec_slippage_bps", "is_bps", "total_cost",
)

### Reading the metrics

- **`fill_rate`** = `total_filled / order_quantity`. Baseline TWAP allocates to every bin including zero-volume ones, so illiquid names (e.g. `PA`) under-fill heavily — the gap a smarter schedule should close.
- **`exec_slippage_bps`** — filled-only slippage vs the arrival price, signed so **positive = worse**.
- **`is_bps`** — implementation shortfall vs arrival price, including opportunity cost on the unfilled remainder marked at the day's terminal price. Captures both spread *and* intraday price drift.

To plug in a new strategy, write a function `my_strategy(bins, quantity, **params) -> Sequence[float]` (per-bin requested quantities) and pass `strategy=my_strategy` to `run_strategy` / `run_trade_list`.